# Winter-wheat phenology from CLMS HRL Croplands

**The question this notebook answers:** can the Copernicus HRL Croplands suite give a
per-cell *winter wheat* sowing/emergence and harvest date, good enough to replace the
SAGE climatology that `site/calendar.py` currently samples?

Three layers, all 10 m, EPSG:3035, EEA reference grid, 100x100 km tiles, 2017-2024:

| Product | Content |
| --- | --- |
| `CTY` Crop Types | 20 classes; wheat is `1110` |
| `CPMCE` Main Crop Emergence | emergence date as `YYDOY` |
| `CPMCH` Main Crop Harvest | harvest date as `YYDOY` |

**The obstacle.** `CTY` does not separate winter from spring wheat -- class `1110` is
just "wheat".

**The hypothesis under test.** `CPMCE` encodes the emergence date as `YYDOY` *including
the year*, precisely because a season labelled 2023 can emerge in 2022. So for wheat
pixels of season year `Y`, emergence should be **bimodal**: a winter mode in the autumn
of `Y-1` and a spring mode inside `Y`. If that split is clean, the winter/spring
separation is an *observation* rather than an assumption, and it is available at 10 m
everywhere the product covers.

Everything downstream of that -- aggregating to the 0.1 degree target grid, comparing
against SAGE -- is only worth building if the bimodality holds. So this notebook tests
it on one small AOI before any pipeline code is written.

**Prerequisites.** A CLMS service key at `~/.clms` (created from the CLMS portal), and an
environment with `rasterio` / `rioxarray` / `pyjwt` (`dev_env` on this machine).

## 1. Setup

The AOI is Brandenburg -- the same area as the existing Germany smoke test, so anything
found here can be checked against `cropmodelling4eu/evaluation/germany_smoke_evaluation.ipynb`.

In [1]:
from __future__ import annotations

import io
import json
import time
import zipfile
from dataclasses import asdict, dataclass
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API = "https://land.copernicus.eu/api"
KEY_FILE = Path.home() / ".clms"

# Brandenburg. Kept small on purpose: one FME job, a few hundred MB at most.
AOI_W, AOI_S, AOI_E, AOI_N = 12.0, 52.0, 13.5, 53.0
SEASON_YEAR = 2023

WORK = Path("outputs/clms_phenology")
WORK.mkdir(parents=True, exist_ok=True)

# CTY class values (from the CDSE STAC `classification:classes` of the CTY collection).
CTY_CLASSES = {
    0: "no cropland", 1110: "wheat", 1120: "barley", 1130: "maize", 1140: "rice",
    1150: "other cereals", 1210: "fresh vegetables", 1220: "dry pulses",
    1310: "potatoes", 1320: "sugar beet", 1410: "sunflower", 1420: "soybeans",
    1430: "rapeseed", 1440: "flax cotton and hemp", 2100: "grapes", 2200: "olives",
    2310: "fruits", 2320: "nuts", 3100: "unclassified annual crop",
    3200: "unclassified permanent crop", 65535: "outside area",
}
WHEAT = 1110

# CPMCE / CPMCH sentinels. Everything >= 65526 is a flag, not a date; 0 is "no annual
# cropland". Aggregating any of these as if it were a DOY is the obvious way to get a
# plausible-looking wrong answer, so they are masked before anything else happens.
DATE_FLAGS = {
    0: "no annual cropland", 65526: "fallow land", 65527: "no field geometry",
    65531: "insufficient observations", 65532: "no season delineation",
    65533: "season outside timeframe", 65535: "outside area",
}

## 2. Authenticate

The CLMS service key is an RSA private key plus the claims needed to sign a JWT
authorization grant, which is exchanged for a one-hour bearer token.

In [2]:
import jwt


def clms_token(key_file: Path = KEY_FILE) -> str:
    """Exchange the service key's signed JWT grant for a bearer token."""
    cfg = json.loads(key_file.read_text())
    now = int(time.time())
    grant = jwt.encode(
        {"iss": cfg["client_id"], "sub": cfg["user_id"], "aud": cfg["token_uri"],
         "iat": now, "exp": now + 3600},
        cfg["private_key"], algorithm="RS256",
    )
    r = requests.post(
        cfg["token_uri"],
        data={"grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer", "assertion": grant},
        headers={"Accept": "application/json"}, timeout=60,
    )
    r.raise_for_status()
    return r.json()["access_token"]


TOKEN = clms_token()
AUTH = {"Accept": "application/json", "Authorization": f"Bearer {TOKEN}"}
print("token acquired")

token acquired


## 3. Discover the datasets

A download needs two identifiers: the dataset `UID` and the
`DatasetDownloadInformationID` of the particular collection within it (each product also
publishes a separate confidence layer, which is a *different* download-information ID
under the same dataset).

In [3]:
WANTED = {
    "CTY":   "Crop Types 2017-present (raster 10 m), Europe, yearly",
    "CPMCE": "Main Crop Emergence 2017-present (raster 10 m), Europe, yearly",
    "CPMCH": "Main Crop Harvest 2017-present (raster 10 m), Europe, yearly",
}


@dataclass(slots=True)
class DatasetRef:
    code: str
    title: str
    uid: str
    download_id: str
    collection: str
    full_format: str


def discover(wanted: dict[str, str]) -> dict[str, DatasetRef]:
    r = requests.get(
        f"{API}/@search",
        params={"portal_type": "DataSet", "metadata_fields": ["UID", "dataset_download_information"],
                "b_size": 400},
        headers=AUTH, timeout=120,
    )
    r.raise_for_status()
    by_title = {it["title"]: it for it in r.json()["items"]}

    refs: dict[str, DatasetRef] = {}
    for code, title in wanted.items():
        it = by_title[title]
        # The dataset record from @search carries only the UID; the download-information
        # block lives on the dataset object itself.
        obj = requests.get(it["@id"], headers=AUTH, timeout=120).json()
        items = obj["dataset_download_information"]["items"]
        # Take the primary RASTER collection, never the confidence layer.
        primary = next(x for x in items if "confidence" not in x["collection"].lower())
        refs[code] = DatasetRef(code, title, it["UID"], primary["@id"],
                                primary["collection"], primary["full_format"])
    return refs


REFS = discover(WANTED)
# asdict, not vars: DatasetRef is slots=True and so has no __dict__.
pd.DataFrame([asdict(r) for r in REFS.values()])

,code,title,uid,download_id,collection,full_format
0,CTY,"Crop Types 2017-present (raster 10 m), Europe,...",0d4d0d517b824d22857e43c4ea6073c9,869dd60b-1cd7-414b-8281-4f5e4eaea8f1,Crop Types,Geotiff
1,CPMCE,Main Crop Emergence 2017-present (raster 10 m)...,325ff9b20fd648cbaee5cd4258f54604,7aaf78c5-43d0-46a0-bdc1-fd178d05c0a6,Main Crop Emergence,Geotiff
2,CPMCH,"Main Crop Harvest 2017-present (raster 10 m), ...",461193fa864d4a6d92de66be9111a4f5,89d9c3a4-8fca-4911-ae8c-948f0eddba8b,Main Crop Harvest,Geotiff


### What formats and projections will it give back?

Both `CTY` and `CPMCE`/`CPMCH` are categorical in the sense that matters here: one is a
class code, the other packs two fields into `YYDOY` and reserves the high values as
flags. Neither survives interpolation -- averaging class `1110` with `1130` gives `1120`,
"barley", and averaging `22300` with `23100` gives a date in the wrong year. Only
nearest-neighbour is safe on either.

So the request asks for `EPSG:3035`, the CRS all three are natively published in, to make
any resampling *unnecessary*.

**That is a hope about the FME job, not a guarantee from it.** The service clips
server-side, and nothing in the API contract promises the clip lands on the source grid
rather than a freshly-aligned one. So the next cell verifies it instead of assuming it:
native 10 m resolution, `EPSG:3035`, and — the part a shape comparison misses entirely —
an *identical affine transform* across all three layers. Two rasters can share a shape
and still sit half a pixel apart, and every step downstream pairs these three arrays
element-wise, so a silent offset would mask emergence dates with the wrong crop type and
still produce a perfectly plausible-looking histogram.

In [4]:
for code, ref in REFS.items():
    pr = requests.get(f"{API}/@projections", params={"uid": ref.uid}, headers=AUTH, timeout=60)
    fc = requests.get(f"{API}/@format_conversion_table", params={"uid": ref.uid}, headers=AUTH, timeout=60)
    print(f"--- {code}")
    print("   projections:", pr.json() if pr.ok else f"HTTP {pr.status_code}")
    print("   formats:", str(fc.json())[:300] if fc.ok else f"HTTP {fc.status_code}")

--- CTY
   projections: ['EPSG:4326', 'EPSG:3035', 'EPSG:3857', 'EPSG:4258', 'EPSG:32738', 'EPSG:32740', 'EPSG:32620', 'EPSG:32622']
   formats: {'GDB': {'GDB': True, 'GML': True, 'GPKG': True, 'Geojson': True, 'Geotiff': False, 'Netcdf': False, 'Shapefile': True, 'WFS': False}, 'GML': {'GDB': False, 'GML': True, 'GPKG': False, 'Geojson': False, 'Geotiff': False, 'Netcdf': False, 'Shapefile': False, 'WFS': False}, 'GPKG': {'GDB': True, 'GML'
--- CPMCE
   projections: ['EPSG:4326', 'EPSG:3035', 'EPSG:3857', 'EPSG:4258', 'EPSG:32738', 'EPSG:32740', 'EPSG:32620', 'EPSG:32622']
   formats: {'GDB': {'GDB': True, 'GML': True, 'GPKG': True, 'Geojson': True, 'Geotiff': False, 'Netcdf': False, 'Shapefile': True, 'WFS': False}, 'GML': {'GDB': False, 'GML': True, 'GPKG': False, 'Geojson': False, 'Geotiff': False, 'Netcdf': False, 'Shapefile': False, 'WFS': False}, 'GPKG': {'GDB': True, 'GML'
--- CPMCH
   projections: ['EPSG:4326', 'EPSG:3035', 'EPSG:3857', 'EPSG:4258', 'EPSG:32738', 'EPSG:32740',

## 4. Request the clipped rasters

Two details of `@datarequest_post` that are easy to get wrong:

- **`BoundingBox` coordinate order.** The prose says `[N, E, S, W]` but the worked
  example in the same page is `[2.354, 46.852, 4.639, 45.882]` for a French AOI --
  longitude first, i.e. `[W, N, E, S]`. The example has to win: read as `[N, E, S, W]`
  those numbers put northern France at latitude 2.35. The returned raster bounds are
  checked against the AOI below, so a wrong guess shows up immediately rather than
  silently returning the wrong place.
- **`TemporalFilter` is epoch milliseconds**, and the portal caps a single request at
  366 days -- fine for a yearly product, one request per season year.

In [5]:
def epoch_ms(d: date) -> int:
    return int(time.mktime(d.timetuple()) * 1000)


def submit(refs: dict[str, DatasetRef], year: int) -> str:
    payload = {"Datasets": [
        {
            "DatasetID": ref.uid,
            "DatasetDownloadInformationID": ref.download_id,
            "OutputFormat": "Geotiff",
            "OutputGCS": "EPSG:3035",
            # [W, N, E, S] -- see the note above.
            "BoundingBox": [AOI_W, AOI_N, AOI_E, AOI_S],
            "TemporalFilter": {"StartDate": epoch_ms(date(year, 1, 1)),
                               "EndDate": epoch_ms(date(year, 12, 31))},
        }
        for ref in refs.values()
    ]}
    r = requests.post(f"{API}/@datarequest_post", json=payload,
                      headers={**AUTH, "Content-Type": "application/json"}, timeout=120)
    r.raise_for_status()
    body = r.json()
    if body.get("ErrorTaskIds"):
        raise RuntimeError(f"request rejected: {body['ErrorTaskIds']}")
    return body["TaskIds"][0]["TaskID"]


# A task for Brandenburg 2023 was already submitted while this notebook was written.
# Set this to None to queue a fresh one; each submission occupies a slot on a shared
# FME queue, so re-running the notebook should not re-request the same clip.
EXISTING_TASK_ID = "66068647426"

TASK_ID = EXISTING_TASK_ID or submit(REFS, SEASON_YEAR)
print("TaskID:", TASK_ID)

TaskID: 66068647426


### Poll until ready

The portal also emails when the job finishes; polling `@datarequest_status_get` gets the
same `DownloadURL` without leaving the notebook. Jobs of this size typically take a few
minutes, but the queue is shared, so the timeout is generous.

In [6]:
def wait(task_id: str, timeout_s: int = 3600, every_s: int = 30) -> dict:
    deadline = time.time() + timeout_s
    seen = None
    while time.time() < deadline:
        r = requests.get(f"{API}/@datarequest_status_get", params={"TaskID": task_id},
                         headers=AUTH, timeout=60)
        r.raise_for_status()
        st = r.json()
        # The endpoint answers either the single task or a {TaskID: task} mapping.
        # Verified against a live task: the response is flat, with `Status` at the top
        # level. The mapping form is kept as a fallback.
        task = st if "Status" in st else st.get(str(task_id), st)
        status = task.get("Status")
        if status != seen:
            print(f"[{time.strftime('%H:%M:%S')}] {status} -- {task.get('Message', '')}")
            seen = status
        if status == "Finished_ok":
            return task
        if status and status.startswith(("Finished_nok", "Cancelled", "Rejected")):
            raise RuntimeError(f"download failed: {task}")
        time.sleep(every_s)
    raise TimeoutError(f"task {task_id} not finished after {timeout_s}s")


TASK = wait(TASK_ID)
print("size: %.1f MB" % (TASK["FileSize"] / 1e6))
print("url:", TASK["DownloadURL"])

[14:08:12] Queued -- 


HTTPError: 401 Client Error: Unauthorized for url: https://land.copernicus.eu/api/@datarequest_status_get?TaskID=66068647426

In [ ]:
ZIP_PATH = WORK / f"clms_{SEASON_YEAR}_task{TASK_ID}.zip"
if not ZIP_PATH.exists():
    with requests.get(TASK["DownloadURL"], stream=True, timeout=1800) as r:
        r.raise_for_status()
        with ZIP_PATH.open("wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)

EXTRACT = WORK / f"{SEASON_YEAR}"
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT)

tifs = sorted(EXTRACT.rglob("*.tif"))
for t in tifs:
    print(f"{t.stat().st_size / 1e6:8.1f} MB  {t.relative_to(EXTRACT)}")

## 5. Load and decode

`YYDOY` unpacks as `YY = value // 1000`, `DOY = value % 1000`. Rather than keep a raw
DOY -- which wraps at New Year and makes "the mean of DOY 300 and DOY 40" a meaningless
number -- every date is converted to an **absolute date** and then expressed as days
since an anchor of **1 August of `SEASON_YEAR - 1`**. On that axis a winter-wheat
emergence lands near +30..+120 and its harvest near +330..+380, both monotone, so an
ordinary median aggregates them correctly and no circular statistics are needed.

In [ ]:
import rioxarray  # noqa: F401
import xarray as xr


def open_layer(code: str) -> xr.DataArray:
    """Open the single-band raster whose filename carries the product code."""
    matches = [t for t in tifs if f"_{code}_" in t.name.upper()]
    if not matches:
        raise FileNotFoundError(f"no raster for {code} in {EXTRACT}")
    da = rioxarray.open_rasterio(matches[0], masked=False).squeeze("band", drop=True)
    da.name = code.lower()
    return da


ANCHOR = date(SEASON_YEAR - 1, 8, 1)


def decode_yydoy(da: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray]:
    """Return (days since ANCHOR, flag code). Non-dates come back as NaN days."""
    v = da.astype("int32")
    is_date = (v > 0) & (v < 65526)
    yy = xr.where(is_date, v // 1000, 0)
    doy = xr.where(is_date, v % 1000, 1)

    # Absolute day number, built without a Python-level date loop: days from ANCHOR to
    # 1 Jan of the emergence year, plus DOY-1.
    years = np.unique(yy.values[is_date.values]) if is_date.any() else np.array([], dtype=int)
    offset = xr.zeros_like(yy)
    for y in years:
        jan1 = date(2000 + int(y), 1, 1)
        offset = xr.where(yy == y, (jan1 - ANCHOR).days, offset)

    days = xr.where(is_date, offset + doy - 1, np.nan).astype("float32")
    flag = xr.where(is_date, 0, v).astype("int32")
    return days, flag


cty = open_layer("CTY")
cpmce_raw = open_layer("CPMCE")
cpmch_raw = open_layer("CPMCH")

LAYERS = {"CTY": cty, "CPMCE": cpmce_raw, "CPMCH": cpmch_raw}
for code, da in LAYERS.items():
    print(f"{code:6s} shape={str(da.shape):16s} crs={da.rio.crs} "
          f"res={tuple(round(v, 3) for v in da.rio.resolution())} dtype={da.dtype}")

In [ ]:
def check_grids(layers: dict[str, xr.DataArray], expect_epsg: int = 3035,
                expect_res: float = 10.0) -> None:
    """Fail loudly if the service did not return the native, mutually aligned grid."""
    ref_code, ref = next(iter(layers.items()))

    problems: list[str] = []
    for code, da in layers.items():
        if da.rio.crs.to_epsg() != expect_epsg:
            problems.append(f"{code}: CRS is {da.rio.crs} not EPSG:{expect_epsg} "
                            "-- the raster was reprojected")
        if not all(abs(abs(v) - expect_res) < 1e-6 for v in da.rio.resolution()):
            problems.append(f"{code}: resolution {da.rio.resolution()} is not "
                            f"{expect_res} m -- the raster was resampled")
        # The decisive test: same affine transform, not merely the same shape.
        #
        # rtol=0 is load-bearing. EPSG:3035 eastings are ~4e6, so np.allclose's default
        # rtol=1e-5 tolerates a 40 m disagreement -- four pixels -- and silently passes
        # a misaligned grid. Compare on an absolute tolerance of a thousandth of a pixel.
        tol = expect_res / 1000.0
        if da.shape != ref.shape or not np.allclose(
            np.array(da.rio.transform()).astype(float),
            np.array(ref.rio.transform()).astype(float), rtol=0.0, atol=tol
        ):
            problems.append(f"{code}: grid differs from {ref_code}\n"
                            f"       {code}: shape {da.shape} transform {tuple(da.rio.transform())[:6]}\n"
                            f"       {ref_code}: shape {ref.shape} transform {tuple(ref.rio.transform())[:6]}")

    if problems:
        raise AssertionError(
            "layers are NOT pixel-aligned; element-wise masking below would pair the "
            "wrong pixels:\n  - " + "\n  - ".join(problems)
            + "\n\nFix by aligning explicitly with nearest-neighbour before masking, e.g.\n"
              "    cpmce_raw = cpmce_raw.rio.reproject_match(cty, resampling=Resampling.nearest)\n"
              "NEVER with the default bilinear -- see the note above."
        )
    print(f"OK: all {len(layers)} layers on the native EPSG:{expect_epsg} "
          f"{expect_res:g} m grid, identical transforms")
    print("   transform:", tuple(round(v, 2) for v in tuple(ref.rio.transform())[:6]))
    print("   bounds:   ", tuple(round(v, 1) for v in ref.rio.bounds()))


check_grids(LAYERS)

emergence_days, emergence_flag = decode_yydoy(cpmce_raw)
harvest_days, harvest_flag = decode_yydoy(cpmch_raw)

### Sanity checks before believing anything

If the bounding-box order was wrong, the reprojected bounds will not contain the AOI and
this cell says so.

In [ ]:
from pyproj import Transformer

to_3035 = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
x0, y0 = to_3035.transform(AOI_W, AOI_S)
x1, y1 = to_3035.transform(AOI_E, AOI_N)
left, bottom, right, top = cty.rio.bounds()
inside = left <= x0 and bottom <= y0 and right >= x1 and top >= y1
print(f"AOI in EPSG:3035: x {x0:.0f}..{x1:.0f}  y {y0:.0f}..{y1:.0f}")
print(f"raster bounds   : x {left:.0f}..{right:.0f}  y {bottom:.0f}..{top:.0f}")
print("AOI covered:", inside)
assert inside, "returned raster does not cover the AOI -- check the BoundingBox order"

In [ ]:
counts = pd.Series(cty.values.ravel()).value_counts()
crop_mix = (
    pd.DataFrame({"pixels": counts})
    .assign(name=lambda d: d.index.map(lambda v: CTY_CLASSES.get(int(v), f"<{v}>")),
            pct=lambda d: 100 * d.pixels / d.pixels.sum())
    .sort_values("pixels", ascending=False)
)
crop_mix.head(12)

In [ ]:
flag_mix = (
    pd.Series(emergence_flag.values.ravel()).value_counts()
    .rename_axis("flag").to_frame("pixels")
    .assign(name=lambda d: d.index.map(lambda v: DATE_FLAGS.get(int(v), "valid date" if v == 0 else f"<{v}>")),
            pct=lambda d: 100 * d.pixels / d.pixels.sum())
)
# flag 0 here means "decoded to a real date"; the CPMCE value 0 ("no annual cropland")
# also lands in the same bucket, so it is separated out explicitly.
no_cropland = int((cpmce_raw == 0).sum())
print(f"pixels with CPMCE == 0 (no annual cropland): {no_cropland:,}")
flag_mix

## 6. The hypothesis test

For wheat pixels of season `SEASON_YEAR`, in which calendar year does emergence fall?

If the winter/spring split is real, this histogram is **bimodal**: an autumn peak in
`SEASON_YEAR - 1` and a spring peak in `SEASON_YEAR`. If instead everything piles into a
single spring mode, then `CPMCE` is reporting post-dormancy regrowth rather than true
autumn emergence -- which would make it no better than HR-VPP's `SOSD`, and the whole
approach collapses back to needing an external winter/spring wheat map.

In [ ]:
wheat = cty == WHEAT
valid = wheat & np.isfinite(emergence_days)
print(f"wheat pixels: {int(wheat.sum()):,}  ({100 * float(wheat.mean()):.1f}% of AOI)")
print(f"  ... with a decoded emergence date: {int(valid.sum()):,} "
      f"({100 * int(valid.sum()) / max(int(wheat.sum()), 1):.1f}%)")

wd = emergence_days.values[valid.values]
hd = harvest_days.values[(wheat & np.isfinite(harvest_days)).values]
print(f"emergence, days since {ANCHOR}: min {wd.min():.0f}  median {np.median(wd):.0f}  max {wd.max():.0f}")

In [ ]:
import matplotlib.pyplot as plt

emergence_year = (cpmce_raw.where(valid) // 1000 + 2000)
year_counts = pd.Series(emergence_year.values.ravel()).dropna().astype(int).value_counts().sort_index()
print(year_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].hist(wd, bins=np.arange(np.floor(wd.min()), np.ceil(wd.max()) + 5, 5),
             color="#4C72B0", edgecolor="none")
axes[0].set_xlabel(f"emergence, days since {ANCHOR}")
axes[0].set_ylabel("wheat pixels")
axes[0].set_title(f"CPMCE, wheat (CTY={WHEAT}), season {SEASON_YEAR}")
for lab, off in [("1 Oct", (date(SEASON_YEAR - 1, 10, 1) - ANCHOR).days),
                 ("1 Jan", (date(SEASON_YEAR, 1, 1) - ANCHOR).days),
                 ("1 Apr", (date(SEASON_YEAR, 4, 1) - ANCHOR).days)]:
    axes[0].axvline(off, color="0.35", ls="--", lw=1)
    axes[0].text(off, axes[0].get_ylim()[1] * 0.94, f" {lab}", fontsize=9, color="0.35")

axes[1].hist(hd, bins=60, color="#55A868", edgecolor="none")
axes[1].set_xlabel(f"harvest, days since {ANCHOR}")
axes[1].set_title(f"CPMCH, wheat, season {SEASON_YEAR}")

fig.tight_layout()

### Split, and check the split against season length

The cut is placed at 1 January of the season year: emergence before it is winter wheat,
after it is spring wheat.

The split is only trustworthy if it survives an *independent* check, so the season length
`harvest - emergence` is the test. Winter wheat in Brandenburg should run roughly
250-320 days; spring wheat roughly 100-160. If both classes return the same season
length, the "winter" mode is not winter wheat and the split is an artefact.

In [ ]:
CUT = (date(SEASON_YEAR, 1, 1) - ANCHOR).days

season_len = harvest_days - emergence_days
both = wheat & np.isfinite(emergence_days) & np.isfinite(season_len)
is_winter = both & (emergence_days < CUT)
is_spring = both & (emergence_days >= CUT)

rows = []
for label, m in [("winter (emerged before 1 Jan)", is_winter), ("spring (emerged after 1 Jan)", is_spring)]:
    e = emergence_days.values[m.values]
    s = season_len.values[m.values]
    if e.size == 0:
        continue
    rows.append({
        "class": label, "pixels": int(m.sum()), "pct_of_wheat": 100 * int(m.sum()) / int(both.sum()),
        "emergence_median": float(np.median(e)),
        "emergence_date": str(ANCHOR + timedelta(days=float(np.median(e)))),
        "season_len_median": float(np.median(s)),
        "season_len_p10": float(np.percentile(s, 10)),
        "season_len_p90": float(np.percentile(s, 90)),
    })
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for label, m, c in [("winter", is_winter, "#4C72B0"), ("spring", is_spring, "#C44E52")]:
    s = season_len.values[m.values]
    if s.size:
        ax.hist(s, bins=np.arange(0, 400, 5), alpha=0.65, label=f"{label} (n={s.size:,})", color=c)
ax.set_xlabel("season length, harvest - emergence (days)")
ax.set_ylabel("wheat pixels")
ax.set_title("Does the emergence-based split separate two real season lengths?")
ax.legend()
fig.tight_layout()

## 7. Aggregate to the 0.1 degree target grid

Following the ECIRA precedent in `CLAUDE.md`: bin the 10 m pixel **centres** into the
target cells rather than resampling, so no pixel is split and no date is interpolated.
The per-cell statistic is the **median** over winter-wheat pixels, on the
days-since-anchor axis.

Two extra fields travel with each cell, mirroring how the soil stage carries
`share_percent`: the number of contributing pixels, and the winter share of that cell's
wheat. A cell whose date rests on a handful of pixels is not the same evidence as one
resting on thousands, and the export should be able to say so.

In [ ]:
GRID_RES = 0.1

to_4326 = Transformer.from_crs(cty.rio.crs, "EPSG:4326", always_xy=True)
xs, ys = np.meshgrid(cty.x.values, cty.y.values)
m = is_winter.values
lon, lat = to_4326.transform(xs[m], ys[m])

cell_lon = np.floor(lon / GRID_RES) * GRID_RES + GRID_RES / 2
cell_lat = np.floor(lat / GRID_RES) * GRID_RES + GRID_RES / 2

per_pixel = pd.DataFrame({
    "cell_lon": np.round(cell_lon, 4),
    "cell_lat": np.round(cell_lat, 4),
    "emergence": emergence_days.values[m],
    "harvest": harvest_days.values[m],
})

cells = (
    per_pixel.groupby(["cell_lat", "cell_lon"])
    .agg(emergence_days=("emergence", "median"),
         harvest_days=("harvest", "median"),
         n_pixels=("emergence", "size"))
    .reset_index()
)

# Back to calendar dates and to the DOY the site stage speaks in.
cells["emergence_date"] = [ANCHOR + timedelta(days=float(d)) for d in cells.emergence_days]
cells["harvest_date"] = [ANCHOR + timedelta(days=float(d)) if np.isfinite(d) else pd.NaT
                         for d in cells.harvest_days]
cells["emergence_doy"] = [d.timetuple().tm_yday for d in cells.emergence_date]
cells["harvest_doy"] = [d.timetuple().tm_yday if pd.notna(d) else np.nan for d in cells.harvest_date]
cells["season_length_days"] = cells.harvest_days - cells.emergence_days

# Winter share of the cell's wheat: 1 - this is the fraction the exported date ignores.
wheat_lon, wheat_lat = to_4326.transform(xs[both.values], ys[both.values])
wheat_cells = (
    pd.DataFrame({"cell_lon": np.round(np.floor(wheat_lon / GRID_RES) * GRID_RES + GRID_RES / 2, 4),
                  "cell_lat": np.round(np.floor(wheat_lat / GRID_RES) * GRID_RES + GRID_RES / 2, 4)})
    .value_counts().rename("n_wheat").reset_index()
)
cells = cells.merge(wheat_cells, on=["cell_lat", "cell_lon"], how="left")
cells["winter_share"] = cells.n_pixels / cells.n_wheat

print(f"{len(cells)} target cells with winter wheat")
cells.sort_values("n_pixels", ascending=False).head(10)

### How thin is the evidence?

A 0.1 degree cell is ~74 km<sup>2</sup> here, i.e. ~740,000 pixels of 10 m. A cell whose
winter-wheat date rests on a few hundred of them is describing a couple of fields.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(cells.n_pixels, bins=40, color="#4C72B0")
axes[0].set_xlabel("winter-wheat pixels per 0.1 deg cell")
axes[0].set_ylabel("cells")
axes[0].set_yscale("log")

sc = axes[1].scatter(cells.cell_lon, cells.cell_lat, c=cells.emergence_doy,
                     s=26, cmap="viridis")
axes[1].set_title("median emergence DOY")
fig.colorbar(sc, ax=axes[1])

sc2 = axes[2].scatter(cells.cell_lon, cells.cell_lat, c=cells.winter_share,
                      s=26, cmap="magma", vmin=0, vmax=1)
axes[2].set_title("winter share of the cell's wheat")
fig.colorbar(sc2, ax=axes[2])

for ax in axes[1:]:
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
fig.tight_layout()

## 8. Compare against SAGE

The point of comparison. SAGE is a 0.5 degree climatology assembled from census
reporting units; CLMS is a 10 m observation of one particular year. They should agree
in the large and disagree in the detail -- and the *shape* of the disagreement is what
decides whether CLMS is worth wiring into `site/calendar.py`.

Remember the offset being compared: SAGE `plant` is a **sowing** date, CPMCE is an
**emergence** date. Autumn germination takes roughly 10-20 days, so CLMS emergence
should sit *later* than SAGE planting by about that much. A near-zero median difference
would actually be the suspicious result.

In [ ]:
SAGE = Path("/data01/FDS/muduchuru/Data/Agri/SAGE_crop_calendar/Wheat.Winter.crop.calendar.fill.nc")

sage = xr.open_dataset(SAGE)
print(list(sage.data_vars))

lat_name = "latitude" if "latitude" in sage.coords else "lat"
lon_name = "longitude" if "longitude" in sage.coords else "lon"

sampled = sage.sel({lat_name: xr.DataArray(cells.cell_lat.values, dims="cell"),
                    lon_name: xr.DataArray(cells.cell_lon.values, dims="cell")},
                   method="nearest")

cmp = cells.assign(
    sage_plant=sampled["plant"].values,
    sage_plant_start=sampled["plant.start"].values if "plant.start" in sage else np.nan,
    sage_plant_end=sampled["plant.end"].values if "plant.end" in sage else np.nan,
    sage_harvest=sampled["harvest"].values,
)


def doy_delta(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Signed difference a - b in days, wrapped into (-182, 183]."""
    d = (np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) % 365.0
    return np.where(d > 182.5, d - 365.0, d)


cmp["emergence_minus_sage_plant"] = doy_delta(cmp.emergence_doy, cmp.sage_plant)
cmp["harvest_minus_sage_harvest"] = doy_delta(cmp.harvest_doy, cmp.sage_harvest)

cmp[["emergence_doy", "sage_plant", "emergence_minus_sage_plant",
     "harvest_doy", "sage_harvest", "harvest_minus_sage_harvest"]].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, col, title in [
    (axes[0], "emergence_minus_sage_plant", "CLMS emergence - SAGE planting"),
    (axes[1], "harvest_minus_sage_harvest", "CLMS harvest - SAGE harvest"),
]:
    d = cmp[col].dropna()
    ax.hist(d, bins=40, color="#4C72B0", edgecolor="none")
    ax.axvline(0, color="k", lw=1)
    ax.axvline(d.median(), color="#C44E52", lw=2, label=f"median {d.median():+.0f} d")
    ax.set_xlabel("days"); ax.set_title(title); ax.legend()
fig.tight_layout()

## 9. Verdict

Fill this in from the numbers above -- the four things that decide whether to build the
pipeline stage:

1. **Bimodality.** Does the wheat emergence histogram show a distinct autumn mode in
   `SEASON_YEAR - 1`? What fraction of wheat pixels is winter?
2. **Season length.** Do the two modes give distinguishable season lengths (~250-320 d
   winter vs ~100-160 d spring)? If not, the split is an artefact and the approach fails.
3. **Coverage.** What fraction of wheat pixels carry a usable date rather than a flag,
   and how many 0.1 degree cells end up with enough pixels to trust?
4. **SAGE offset.** Is `CLMS emergence - SAGE planting` positive and of plausible
   germination magnitude (~10-20 d), or is it large/erratic in a way that says one of the
   two is wrong?

If 1-3 hold, the next steps are:

- Repeat for 2017-2024 to see how stable the per-cell dates are between years -- that
  interannual spread is the honest error bar on any climatology built from them, and it is
  something SAGE cannot provide at all.
- Widen to a second AOI with a different cropping regime (southern Spain, or Denmark) to
  check the 1 January cut generalises. It almost certainly needs to be latitude-dependent
  rather than fixed.
- Decide sowing-from-emergence: back off a degree-day germination requirement using the
  MSWX temperature the pipeline already loads, and label it in `calendar_source` the way
  the SAGE path already labels `nearest` / `fallback`.
- Only then wire `calendar_source: clms` into `site/calendar.py`.

An independent observed phenology also resolves the circularity warning on
`sage_calendar_evaluation.ipynb`, and gives the rule-based sowing window something real
to be calibrated against.

---

## Appendix: the other access route

`@datarequest_post` is convenient because it clips server-side, but it is an
asynchronous FME job on a shared queue -- fine for an AOI, questionable for 862 tiles
x 8 years x 3 layers.

For a bulk run there is a public, unauthenticated manifest of every tile, with the CDSE
product UUID, the S3 path, size and MD5. Fetching whole 100x100 km tiles and clipping
locally sidesteps the queue entirely. The tiles are small -- CTY averages 3.6 MB,
CPMCE/CPMCH about 7 MB -- so all three layers for all eight years over the European
domain is on the order of 120 GB.

Downloading from the UUID needs a **CDSE** account (OIDC or S3 keys), which is a
different credential from the CLMS service key used above.

In [ ]:
MANIFEST = ("https://s3.waw3-1.cloudferro.com/swift/v1/CatalogueCSV/landcover_landuse/"
            "cropping_patterns/clms_vlcc_main-crop-emergence_europe_10m_yearly_v1/"
            "clms_vlcc_main-crop-emergence_europe_10m_yearly_v1_cog.csv")

idx = pd.read_csv(MANIFEST, sep=";")
idx["year"] = pd.to_datetime(idx.content_date_start).dt.year
idx["tile"] = idx.name.str.extract(r"_R10m_(E\d+N\d+)_")

print(f"{len(idx):,} tiles, {idx.year.min()}-{idx.year.max()}")
print(idx.groupby("year").agg(tiles=("id", "size"), GB=("content_length", lambda s: s.sum() / 1e9)).round(1))
idx[["name", "tile", "year", "content_length", "s3_path"]].head(3)